structured output
model can be requested to provide response in a formate matching a given schema. This is useful for ensuring the output can be easily parsed and useful in subsequent processing. Lnagcahin supports multiple schema types and methods for enforcing structured output

Pydantic: 
pydantic models provides richest feature set with field validation, nested structures,descriptions

In [1]:
import os
from dotenv import load_dotenv  
from langchain.chat_models import init_chat_model
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")

In [3]:
from pydantic import BaseModel, Field
class movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="The year the movie was released")
    director:str = Field(description="The director of the movie")
    rating:float = Field(description="The rating of the movie out of 10")

In [14]:
model_with_structure=model.with_structured_output(movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F6B840FCB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F6B8624AD0>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The ti

In [15]:

response=model_with_structure.invoke("Tell me about the movie Inception")
print(f"Response: {response}")


Response: title='Inception' year=2010 director='Christopher Nolan' rating=8.8


In [16]:
from pydantic import BaseModel, Field
class movie(BaseModel):
    title:str = Field(..., description="The title of the movie")
    year:int = Field(..., description="The year the movie was released")
    director:str = Field(..., description="The director of the movie")
    rating:float = Field(..., description="The rating of the movie out of 10")

model_with_structure=model.with_structured_output(movie,include_raw=True)
response=model_with_structure.invoke("Tell me about the movie Inception")
print(f"Response: {response}")

Response: {'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "Tell me about the movie Inception". We need to provide information. Could use the function to retrieve data? The function movie takes director, rating, title, year. We can call it with Inception details. Provide description. Let\'s call function.', 'tool_calls': [{'id': 'fc_7d3b0a97-77ca-4a47-b0d7-3424304e11f6', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 108, 'prompt_tokens': 160, 'total_tokens': 268, 'completion_time': 0.234755446, 'completion_tokens_details': {'reasoning_tokens': 56}, 'prompt_time': 0.007468042, 'prompt_tokens_details': None, 'queue_time': 0.348895713, 'total_time': 0.242223488}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_068241849b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 

NESTED STRUCTURES

In [17]:
from pydantic import BaseModel

class Actors(BaseModel):
    name: str
    age: int
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    director: str
    rating: float
    actors: list[Actors]
model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("Tell me about the movie Inception")
print(f"Response: {response}")

Response: title='Inception' year=2010 director='Christopher Nolan' rating=8.8 actors=[Actors(name='Leonardo DiCaprio', age=45, role='Dom Cobb'), Actors(name='Joseph Gordon-Levitt', age=42, role='Arthur'), Actors(name='Elliot Page', age=38, role='Ariadne'), Actors(name='Tom Hardy', age=45, role='Eames'), Actors(name='Ken Watanabe', age=71, role='Saito')]


TypeDicts
TypeDicts provide simple alternative using python inbuilt-typing. It don't do run time validation.It only does tell type should be this even if llm is thowing error we won't getit. But it is enough to inform llm i want output in this formate.

In [22]:
from typing_extensions import Annotated, TypedDict
class movie(TypedDict):
    title: Annotated[str,..., "The title of the movie"]
    year: Annotated[int,..., "The year the movie was released"]
    director: Annotated[str,..., "The director of the movie"]
    rating: Annotated[float,..., "The rating of the movie out of 10"]
model_with_structure=model.with_structured_output(movie)
response=model_with_structure.invoke("Tell me about the movie Inception")
print(f"Response: {response}")


Response: {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}


In [23]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

Dataclasses:
a data class is class typically containing mainly data,although there are not any restriction. we create it using @dataclass decorator

In [24]:
from dataclasses import dataclass

@dataclass
class contactInfo:
    name: str
    email: str
    phone: str  
model_with_structure=model.with_structured_output(contactInfo)
response=model_with_structure.invoke("Provide contact information for John Doe having email john.doe@example.com and phone number 123-456-7890")
print(f"Response: {response}")


Response: {'email': 'john.doe@example.com', 'name': 'John Doe', 'phone': '123-456-7890'}


Applying all the strutured output with normal agent creation for all pydantic and typdict we can use the below code

In [39]:
import os
from dotenv import load_dotenv  
from langchain.agents import create_agent
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
agent=create_agent("groq:openai/gpt-oss-120b", response_format=contactInfo)
response=agent.invoke({"messages":"Provide contact information for John Doe having email john.doe@example.com and phone number 123-456-7890"})
print(f"Response: {response["structured_response"]}")
print(f"Response: {response["structured_response"].name}")



Response: contactInfo(name='John Doe', email='john.doe@example.com', phone='123-456-7890')
Response: John Doe
